# Name : Maulik Sindhva
# Student id : 2434771
# Phase 2 Grid search

In [1]:
!unzip /animal_10.zip

Streaming output truncated to the last 5000 lines.
  inflating: animal_10/raw-img/scoiattolo/OIP-TvOAcQz_YUs_i6JjPxvGOQHaE8.jpeg  
  inflating: __MACOSX/animal_10/raw-img/scoiattolo/._OIP-TvOAcQz_YUs_i6JjPxvGOQHaE8.jpeg  
  inflating: animal_10/raw-img/scoiattolo/OIP-f_tIXmwqyOYdSTQTsYaxnwHaE7.jpeg  
  inflating: __MACOSX/animal_10/raw-img/scoiattolo/._OIP-f_tIXmwqyOYdSTQTsYaxnwHaE7.jpeg  
  inflating: animal_10/raw-img/scoiattolo/OIP-fq4f4c8nMUFl9J0J9XBNSAHaFj.jpeg  
  inflating: __MACOSX/animal_10/raw-img/scoiattolo/._OIP-fq4f4c8nMUFl9J0J9XBNSAHaFj.jpeg  
  inflating: animal_10/raw-img/scoiattolo/OIP-vwx9bSm37ihiVWqQUB3cyQHaEo.jpeg  
  inflating: __MACOSX/animal_10/raw-img/scoiattolo/._OIP-vwx9bSm37ihiVWqQUB3cyQHaEo.jpeg  
  inflating: animal_10/raw-img/scoiattolo/OIP-NJ9pzTUqMLcwj-aAx4Ke4wAAAA.jpeg  
  inflating: __MACOSX/animal_10/raw-img/scoiattolo/._OIP-NJ9pzTUqMLcwj-aAx4Ke4wAAAA.jpeg  
  inflating: animal_10/raw-img/scoiattolo/OIP-6uBuLoN_VRsIyYBUuT7SYQHaE7.jpeg  
  inflating: _

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2, shufflenet_v2_x1_0
from torch.utils.data import DataLoader, random_split
import itertools
import time
import copy



In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Combinations to find best hyperparameters


In [4]:
batch_sizes = [16, 32]
learning_rates = [0.001, 0.0005, 0.0001]
optimizers_list = ["adam", "sgd"]

hyperparameter_combinations = list(itertools.product(batch_sizes, learning_rates, optimizers_list))
# itertools.product()
# It produces all possible combinations from multiple lists.

print(f"Total combinations to test: {len(hyperparameter_combinations)}")
for combo in hyperparameter_combinations:
    print(f"Batch={combo[0]}, LR={combo[1]}, Optimizer={combo[2]}")

Total combinations to test: 12
Batch=16, LR=0.001, Optimizer=adam
Batch=16, LR=0.001, Optimizer=sgd
Batch=16, LR=0.0005, Optimizer=adam
Batch=16, LR=0.0005, Optimizer=sgd
Batch=16, LR=0.0001, Optimizer=adam
Batch=16, LR=0.0001, Optimizer=sgd
Batch=32, LR=0.001, Optimizer=adam
Batch=32, LR=0.001, Optimizer=sgd
Batch=32, LR=0.0005, Optimizer=adam
Batch=32, LR=0.0005, Optimizer=sgd
Batch=32, LR=0.0001, Optimizer=adam
Batch=32, LR=0.0001, Optimizer=sgd


## Data Loading

In [5]:
# ================================
# 1) TRANSFORMS
# ================================

# No augmentation, no normalization
transform_raw = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Augmentation + Normalization (for pretrained MobileNet & ShuffleNet)
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

# When using pretrained models like MobileNet or ShuffleNet,
# the model was originally trained on the ImageNet dataset.
# ImageNet images are RGB, with pixel values between 0 and 1 (after ToTensor())
# Each channel (R, G, B) has a mean and standard deviation calculated over all images.
transform_aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

# ================================
# 2) DATASETS
# ================================

data_dir = "/content/animal_10/raw-img"

dataset_raw = datasets.ImageFolder(root=data_dir, transform=transform_raw)
dataset_aug = datasets.ImageFolder(root=data_dir, transform=transform_aug)

# ================================
# 3) TRAIN/VAL SPLIT
# ================================

train_size = int(0.8 * len(dataset_raw))
val_size = len(dataset_raw) - train_size

train_raw, val_raw = random_split(dataset_raw, [train_size, val_size])
train_aug, val_aug = random_split(dataset_aug, [train_size, val_size])

# ================================
# 4) DATALOADERS
# ================================

# RAW DATA DATALOADERS (no aug, no normalization)
train_loader_raw = DataLoader(train_raw, batch_size=32, shuffle=True)
val_loader_raw   = DataLoader(val_raw,   batch_size=32, shuffle=False)

# AUGMENTED + NORMALIZED DATA DATALOADERS
train_loader_aug = DataLoader(train_aug, batch_size=32, shuffle=True)
val_loader_aug   = DataLoader(val_aug,   batch_size=32, shuffle=False)



## Mobilenet


In [6]:
mobilenet = mobilenet_v2(weights="IMAGENET1K_V1")

# Fully connected last layer should have 10 classes in the end
mobilenet.classifier[1] = nn.Linear(mobilenet.last_channel, 10)

mobilenet = mobilenet.to(device)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 51.8MB/s]


## Shufflenet


In [7]:
shufflenet = shufflenet_v2_x1_0(weights="IMAGENET1K_V1")

# Fully connected last layer should have 10 classes in the end
shufflenet.fc = nn.Linear(shufflenet.fc.in_features, 10)

shufflenet = shufflenet.to(device)

Downloading: "https://download.pytorch.org/models/shufflenetv2_x1-5666bf0f80.pth" to /root/.cache/torch/hub/checkpoints/shufflenetv2_x1-5666bf0f80.pth


100%|██████████| 8.79M/8.79M [00:00<00:00, 39.6MB/s]


## Loss Function


In [8]:
criterion = nn.CrossEntropyLoss()

## Training and Evaluation loops

In [9]:
def train_model(model, train_loader, criterion, optimizer):
    model.train()
    running_loss, total, correct = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (preds == labels).sum().item()

    return running_loss / len(train_loader), 100 * correct / total


def evaluate_model(model, val_loader, criterion):
    model.eval()
    running_loss, total, correct = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    return running_loss / len(val_loader), 100 * correct / total


## Grid Search

In [10]:
def grid_search_with_combinations(model_name, model, train_dataset, val_dataset,
                                  hyperparameter_combinations, num_epochs=3):
    results = []
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for batch, lr, opt_name in hyperparameter_combinations:
        print(f"\n {model_name} | Batch={batch} | LR={lr} | Optimizer={opt_name}")

        # Create dataloaders
        train_loader = DataLoader(train_dataset, batch_size=batch, shuffle=True)
        val_loader   = DataLoader(val_dataset, batch_size=batch, shuffle=False)

        # Use a fresh copy of the model
        model_copy = copy.deepcopy(model).to(device)

        # Define optimizer
        if opt_name == "adam":
            optimizer = optim.Adam(model_copy.parameters(), lr=lr)
        elif opt_name == "sgd":
            optimizer = optim.SGD(model_copy.parameters(), lr=lr, momentum=0.9)
        else:
            raise ValueError("Optimizer must be 'adam' or 'sgd'")

        # Train and evaluate
        for epoch in range(num_epochs):
            train_loss, train_acc = train_model(model_copy, train_loader, criterion, optimizer)
            val_loss, val_acc = evaluate_model(model_copy, val_loader, criterion)
            print(f"Epoch {epoch+1}/{num_epochs} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

        # Save results
        results.append({
            "model": model_name,
            "batch": batch,
            "lr": lr,
            "optimizer": opt_name,
            "val_acc": val_acc
        })

    return results


## Implementation

In [ ]:
# ======= RAW DATA =======
print("===== GRID SEARCH: RAW DATA ======")
results_raw_mobilenet = grid_search_with_combinations("mobilenet", mobilenet, train_raw, val_raw,
                                                      hyperparameter_combinations, num_epochs=3)
results_raw_shufflenet = grid_search_with_combinations("shufflenet", shufflenet, train_raw, val_raw,
                                                       hyperparameter_combinations, num_epochs=3)

# ======= AUGMENTED DATA =======
print("===== GRID SEARCH: AUGMENTED DATA ======")
results_aug_mobilenet = grid_search_with_combinations("mobilenet", mobilenet, train_aug, val_aug,
                                                      hyperparameter_combinations, num_epochs=3)
results_aug_shufflenet = grid_search_with_combinations("shufflenet", shufflenet, train_aug, val_aug,
                                                       hyperparameter_combinations, num_epochs=3)


===== GRID SEARCH: RAW DATA ======

 mobilenet | Batch=16 | LR=0.001 | Optimizer=adam
Epoch 1/3 | Train Acc: 77.14% | Val Acc: 86.55%
Epoch 2/3 | Train Acc: 84.86% | Val Acc: 84.03%
Epoch 3/3 | Train Acc: 86.83% | Val Acc: 88.62%

 mobilenet | Batch=16 | LR=0.001 | Optimizer=sgd
Epoch 1/3 | Train Acc: 91.12% | Val Acc: 95.72%
Epoch 2/3 | Train Acc: 96.03% | Val Acc: 95.89%
Epoch 3/3 | Train Acc: 97.48% | Val Acc: 96.43%

 mobilenet | Batch=16 | LR=0.0005 | Optimizer=adam
Epoch 1/3 | Train Acc: 84.47% | Val Acc: 91.50%
Epoch 2/3 | Train Acc: 89.63% | Val Acc: 89.76%
Epoch 3/3 | Train Acc: 91.91% | Val Acc: 91.23%

 mobilenet | Batch=16 | LR=0.0005 | Optimizer=sgd
Epoch 1/3 | Train Acc: 90.16% | Val Acc: 96.39%
Epoch 2/3 | Train Acc: 96.12% | Val Acc: 96.62%
Epoch 3/3 | Train Acc: 97.38% | Val Acc: 96.93%

 mobilenet | Batch=16 | LR=0.0001 | Optimizer=adam
Epoch 1/3 | Train Acc: 91.16% | Val Acc: 95.57%
Epoch 2/3 | Train Acc: 96.12% | Val Acc: 96.01%
Epoch 3/3 | Train Acc: 97.43% | Val A

In [1]:
# I will continue with the cell output
# Next combination is mobilenet | Batch=32 | LR=0.0001 | Optimizer=sgd

In [11]:
hyperparameter_combinations_1 = list(itertools.product(
    [32],
    [0.0001],
    ["sgd"]
))

print(f"Total combinations to test: {len(hyperparameter_combinations_1)}")
for combo in hyperparameter_combinations_1:
    print(f"Batch={combo[0]}, LR={combo[1]}, Optimizer={combo[2]}")


Total combinations to test: 1
Batch=32, LR=0.0001, Optimizer=sgd


In [ ]:
# Remaining combination for MobileNet
results_aug_mobilenet = grid_search_with_combinations(
    "mobilenet", mobilenet, train_aug, val_aug,
    hyperparameter_combinations_1, num_epochs=3
)

# ALL combinations for ShuffleNet on augmented dataset
results_aug_shufflenet = grid_search_with_combinations(
    "shufflenet", shufflenet, train_aug, val_aug,
    hyperparameter_combinations, num_epochs=3
)


 mobilenet | Batch=32 | LR=0.0001 | Optimizer=sgd
Epoch 1/3 | Train Acc: 74.32% | Val Acc: 91.67%
Epoch 2/3 | Train Acc: 91.33% | Val Acc: 93.89%
Epoch 3/3 | Train Acc: 92.53% | Val Acc: 94.44%

 shufflenet | Batch=16 | LR=0.001 | Optimizer=adam
Epoch 1/3 | Train Acc: 79.81% | Val Acc: 88.66%
Epoch 2/3 | Train Acc: 87.17% | Val Acc: 90.68%
Epoch 3/3 | Train Acc: 89.07% | Val Acc: 90.36%

 shufflenet | Batch=16 | LR=0.001 | Optimizer=sgd
Epoch 1/3 | Train Acc: 29.52% | Val Acc: 36.06%
Epoch 2/3 | Train Acc: 37.41% | Val Acc: 47.69%
Epoch 3/3 | Train Acc: 55.03% | Val Acc: 68.70%

 shufflenet | Batch=16 | LR=0.0005 | Optimizer=adam
Epoch 1/3 | Train Acc: 81.14% | Val Acc: 91.56%
Epoch 2/3 | Train Acc: 90.40% | Val Acc: 93.24%
Epoch 3/3 | Train Acc: 92.03% | Val Acc: 93.14%

 shufflenet | Batch=16 | LR=0.0005 | Optimizer=sgd
Epoch 1/3 | Train Acc: 28.84% | Val Acc: 36.12%
Epoch 2/3 | Train Acc: 36.06% | Val Acc: 36.36%
Epoch 3/3 | Train Acc: 35.95% | Val Acc: 36.40%

 shufflenet | Batch=

In [12]:
hyperparameter_combinations_2 = list(itertools.product(
    [32],
    [0.0001],
    ["adam"]
))

print(f"Total combinations to test: {len(hyperparameter_combinations_1)}")
for combo in hyperparameter_combinations_1:
    print(f"Batch={combo[0]}, LR={combo[1]}, Optimizer={combo[2]}")

# Last 2 combinations for ShuffleNet on augmented dataset
# shufflenet | Batch=32 | LR=0.0001 | Optimizer=adam
# shufflenet | Batch=32 | LR=0.0001 | Optimizer=sgd

results_aug_shufflenet = grid_search_with_combinations(
    "shufflenet", shufflenet, train_aug, val_aug,
    hyperparameter_combinations_2, num_epochs=3
)

results_aug_shufflenet = grid_search_with_combinations(
    "shufflenet", shufflenet, train_aug, val_aug,
    hyperparameter_combinations_1, num_epochs=3
)

Total combinations to test: 1
Batch=32, LR=0.0001, Optimizer=sgd

 shufflenet | Batch=32 | LR=0.0001 | Optimizer=adam
Epoch 1/3 | Train Acc: 66.22% | Val Acc: 88.81%
Epoch 2/3 | Train Acc: 88.89% | Val Acc: 94.02%
Epoch 3/3 | Train Acc: 92.39% | Val Acc: 94.67%

 shufflenet | Batch=32 | LR=0.0001 | Optimizer=sgd
Epoch 1/3 | Train Acc: 17.60% | Val Acc: 19.48%
Epoch 2/3 | Train Acc: 20.57% | Val Acc: 21.89%
Epoch 3/3 | Train Acc: 23.96% | Val Acc: 24.52%
